# 02 - Audit the data and build the split (GW runs this; everyone reads the output)

**Plain words:** before training anything we (1) check the data for cheating routes, (2) remove duplicates, and
(3) decide which images are used for training, for tuning, for testing, and for the *surprise exam* (generators the model has never seen).

**You need:** everyone's `public_*_<CODE>.csv` files in the Drive `metadata/` folder (the pull notebook puts them there).

**Outputs** (saved to Drive `audit_outputs/`): `split.csv`, `recommended.json`, `shortcut_test.csv`, audit tables and charts.
Afterwards upload `split.csv`, `recommended.json`, `shortcut_test.csv` to the GitHub folder `audit_outputs/`.

In [ ]:
# CELL 1 - run this first. It downloads the project code into this Colab session.
import os, subprocess, sys
REPO_URL = "https://github.com/andrinawatsemba/DEEPFAKE_CV_INVESTIGATION.git"
CODE = "/content/repo"
if not os.path.isdir(CODE + "/src"):
    for branch in ("main", "gw/lean-pipeline"):          # main first; the branch is only a fallback
        subprocess.run(["rm", "-rf", CODE])
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO_URL, CODE])
        if r.returncode == 0 and os.path.exists(CODE + "/src/colab_setup.py"):
            print("Got the code from branch:", branch)
            break
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"])
sys.path.insert(0, CODE + "/src")
print("Code ready.")


In [ ]:
# CELL 2 - connect Drive, load everyone's CSVs
!pip -q install -U imagehash
import os, pandas as pd, numpy as np
from colab_setup import setup
import audit_split as A
ROOT, REPO = setup()
META = A.load_all_meta(os.path.join(ROOT, "metadata"))
print(len(META), "images from", META.member_file.nunique(), "metadata files")
display(META.groupby(["member_file", "label"]).size().unstack(fill_value=0))

In [ ]:
# CELL 3 - remove duplicates. Exact copies (same SHA-256) are always removed.
# Near-duplicates (the same picture re-saved) need the images and take a few minutes: set RUN_PHASH = True to include them.
RUN_PHASH = False
phash = A.compute_phash(META, ROOT) if RUN_PHASH else None
DATA, dup_info = A.dedupe(META, phash)
print(dup_info)

In [ ]:
# CELL 4 - THE AUDIT. Look at every table. A gap between real and AI here is a shortcut the model could cheat with.
import matplotlib.pyplot as plt
tabs = A.format_audit(DATA)
for k in ("counts_by_class_source", "format_by_class", "format_share_by_class", "generators"):
    print("\n==", k); display(tabs[k])

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
tabs["format_share_by_class"].T.plot.bar(stacked=True, ax=ax[0]); ax[0].set_title("File format share per class")
for lab, c in (("real", "#14202B"), ("ai", "#A64B14")):
    d = DATA[DATA.label == lab]
    ax[1].hist(np.log10(d.width * d.height), bins=30, alpha=.6, label=lab, color=c)
    ax[2].hist(np.log10(d.bytes_per_pixel.clip(1e-3)), bins=30, alpha=.6, label=lab, color=c)
ax[1].set_title("Image size (log10 pixels)"); ax[2].set_title("Bytes per pixel (log10)"); ax[1].legend()
plt.tight_layout(); os.makedirs(os.path.join(ROOT, "audit_outputs"), exist_ok=True)
plt.savefig(os.path.join(ROOT, "audit_outputs", "audit_overview.png"), dpi=130); plt.show()

In [ ]:
# CELL 5 - THE SHORTCUT TEST: can a model tell real from AI using NO pixels at all (only format, size, file bytes)?
# ROC-AUC 0.5 = no leak. Above 0.75 = shortcut. Read the plain-language guide below the table.
SHORT = A.metadata_shortcut_test(DATA)
display(SHORT)

**How to read the shortcut table**

* `format only` or `width + height` high: the *file type or picture size* gives the answer away. Fix: balance by format (training does this automatically)
  and use the class-blind augmentation (Runs C and E). Write the number in the report.
* Only `file size + bytes per pixel` high: this can be a *real* signal (AI images are often smoother, so they compress smaller) and not only a file-type leak.
  It is exactly why every run re-checks its own training set, and why the format-swap and WhatsApp tests exist.
* All near 0.5: no metadata leak. Good, but still run the format-swap test later.

In [ ]:
# CELL 6 - BUILD THE SPLIT
SPLIT, info = A.make_split(DATA, holdout_frac=0.20, seed=42)
REC = A.recommend(SPLIT)
print("Held-out (unseen) generators:", info["holdout_generators"])
display(SPLIT.groupby(["role", "label"]).size().unstack(fill_value=0))
print("\nRecommended for every run: single generator =", REC["single_generator"], "| images per class =", REC["n_per_class"])
if REC["warning"]:
    print("WARNING:", REC["warning"])

In [ ]:
# CELL 7 - save everything to Drive, then download the small files for GitHub
A.save_all(SPLIT, info, REC, tabs, SHORT, os.path.join(ROOT, "audit_outputs"))
import shutil
from google.colab import files
for f in ("split.csv", "recommended.json", "shortcut_test.csv", "split_info.json"):
    shutil.copy(os.path.join(ROOT, "audit_outputs", f), f)
    files.download(f)
print("Saved to Drive/audit_outputs and downloaded. Upload these files to GitHub: audit_outputs/")